# Two-Port S-Parameter Sweep

Measures S11, S12, S21 and S22 at a chosen set of MM4250 switch positions and saves each one as a `.s2p` file plus a QCoDeS run.

Uses `sweep_db.py` and `vna_measure.py`, which must sit in the **same folder as this notebook**. Outputs land in that folder too:

```
<this folder>/Sweeps/<date>_<temp>/<serials>/raw/<position>.s2p
<this folder>/mm4250_sweeps.db
```

All four S-parameters come from a **single sweep** per position -- `run_sweep()` triggers every trace on the VNA channel, and the P5004B handles the reverse sweep for S12/S22 itself.

Raw acquisition only -- no calibration or de-embedding is applied. These are S-parameters at the VNA's own port reference planes, so what they mean depends on how the VNA is cabled into your setup.

**Instruments:** the cell below connects its own `ksvna`/`switch`. If you'd rather sweep from `QCodesMeasurmentFramework.ipynb`'s kernel, where those already exist, skip that cell and call `run_twoport_sweep(...)` there instead -- it finds instruments registered under those names on its own. Don't do both in one kernel: two live `MM4250("switch")` instances collide, and the switch's USB connection is exclusive.

In [ ]:
import sys
from pathlib import Path

# The drivers live in whichever repo root is above this notebook -- this
# folder if it's the mm4250-switch repo, or the framework repo if these
# files were copied into users/<name>/. Walk up until we find it.
try:
    here = Path(_dh[0])  # Jupyter sets _dh[0] to this notebook's directory
except NameError:
    here = Path.cwd()
if str(here) not in sys.path:
    sys.path.insert(0, str(here))       # vna_measure / sweep_db live here

root = here
while not (root / "drivers").is_dir() and root != root.parent:
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))       # drivers/ live here
print(f"notebook folder: {here}")
print(f"drivers from:    {root}")

from drivers.KeysightVNA_driver import KeysightP5004B
from drivers.MM4250_QCodes_driver import MM4250
from vna_measure import setup_sweep, sweep_settings, measure_2port
from sweep_db import run_twoport_sweep

In [ ]:
ksvna = KeysightP5004B("ksvna", "TCPIP0::QTSF-Measurement::hislip_PXI0_CHASSIS1_SLOT1_INDEX0::INSTR")
switch = MM4250("switch")

## Sweep setup

Only the settings you actually pass are changed, so `setup_sweep(points=2001)` nudges one thing and leaves the rest alone.

Changing the frequency range, point count or IF bandwidth **invalidates any calibration applied on the VNA** -- re-run the cal after changing them.

In [ ]:
setup_sweep(start=1e9, stop=10e9, points=1001, if_bandwidth=1e3, power=-20, vna=ksvna)

## A single measurement

Nothing saved -- `data` is a dict of complex numpy arrays keyed by S-parameter name.

In [ ]:
freq, data = measure_2port(3, vna=ksvna, switch=switch)

import numpy as np
for sparam, values in data.items():
    print(f"{sparam}: {20 * np.log10(np.abs(values)).min():.2f} to "
          f"{20 * np.log10(np.abs(values)).max():.2f} dB")

## Batch sweep

`positions` is a list of RF channel numbers (any subset of 1-6), switch state names as strings, or a mix of both:

| `positions` | what it measures |
|---|---|
| `[1, 3, 5]` | three RF channels |
| `list(range(1, 7))` | all six |
| `[3, "ALL_OPEN"]` | RF3 connected, then the same cabling with everything open |

Mixing the two is how you get isolation without touching a cable: hold the cabling fixed, measure the channel that's connected, then measure again on a state that disconnects it. Valid state names are `"ALL_OPEN"`, `"RFC_RF1"`..`"RFC_RF6"`, `"INTERNAL_LOAD"` and `"INTERNAL_SHORT"`.

Add `prompt_between=True` if something has to be re-cabled by hand between positions -- the sweep then pauses and waits for Enter before each one.

In [ ]:
# Edit these for each run, then save the whole set:
positions = [1, 3, 5]        # channels 1-6, state-name strings, or both
date_str = "YYYYMMDD"        # e.g. "20260920"
temp_str = "295K"            # e.g. "295K", "3K", "25mK"
switch_serials = "SN0001"    # the switch(es) under test, e.g. "0077_0078"

sweep_dir = run_twoport_sweep(positions, date_str, temp_str, switch_serials,
                              vna=ksvna, switch=switch)
print(f"Sweep complete, .s2p files saved under {sweep_dir}")

## Close

Releases both instruments. The MM4250 holds an exclusive USB handle, so run this before starting another notebook that needs the switch.

In [ ]:
ksvna.close()
switch.close()